<a href="https://colab.research.google.com/github/okletsdothis-lang/Delete-all-your-Roblox-badges-/blob/main/badge%20deletion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Roblox Batch Badge Deletion Script

This script is designed to delete *all* your Roblox badges. **Before running, please follow the crucial step below to insert your Roblox security cookie.**

⚠️ **Important:** Roblox may still block deletion depending on current API permissions. This is the correct method, but not every account/game supports it. This script will attempt to delete all badges found for your account.

### &#128272; **CRUCIAL STEP: Add your Roblox Security Cookie (KEEP PRIVATE!)**

**You MUST replace `"YOUR_COOKIE_HERE"` in the code cell below with your actual `.ROBLOSECURITY` cookie value.** This cookie is necessary for authentication. Never share this cookie with anyone as it grants access to your Roblox account.

#### **Option 1: Desktop (Chrome, Edge, Firefox)**
1. Go to `roblox.com` and log in.
2. Open Developer Tools (`F12` or `Ctrl+Shift+I`).
3. Go to the **Application** (Chrome/Edge) or **Storage** (Firefox) tab.
4. Under **Cookies**, select `https://www.roblox.com`.
5. Find the cookie named `.ROBLOSECURITY` and copy the long string in the 'Value' field.

#### **Option 2: iOS (iPhone/iPad)**
1. Download the **Orion Browser** from the App Store.
2. Install the **Cookie-editor** extension within Orion.
3. Log into `roblox.com` in Orion.
4. Open the Cookie-editor extension, find `.ROBLOSECURITY`, and copy its value.

In [ ]:
import requests
import time

# --- CONFIGURATION ---
ROBLOSECURITY = "_|WARNING:-DO-NOT-SHARE-THIS.--Sharing-this-will-allow-someone-to-log-in-as-you-and-to-steal-your-ROBUX-and-items.|_CAEaAhADIhsKBGR1aWQSEzgyMzk5NTU3MjA3NDEzNDcyMzIoAw.ljSUVze0UQe5E5dcxo1ThSINfDtry97TCnxZTv16LSVTvsCwFZborV7W3JMohT7hwTdjutXD03F_O7QnCYo9hz9nA0XXHiVv2ZPfrFKIz6ODI2XS9bpQAluF1PpNkLmEwkGmQbpobeV3SZCbRf3L8AKrMbw5cknihJmbDoj1VHryJkObJLuS-40v7ZKe6pA7khV5_i9bUiQmvbV_KyDPJGFoiopNECjSKXp18kjRQAx_6gDyqCHln123siPpAYNvQ-HptpGlte5mVMaNC76f8aHGnU-SjgHB5-eoT4JRAa5N4IgkI1DOXmUdzBbewgg283msZ1b5o88-8L17nQdIUBoN9MrXmrdfwyclLGhe4VMOsBf9O_9AHdgty4HgVseMVSOZD7YI4HDp7JFrWgqCk2vaW6U-v3MhK0qeFmKA6DgH4SjiToMBGo40h7Jx3r5EKM8xFgwXVdo_GC5kha6nKOsl9YLrVzInOH78eeN8aznVIjY5X9EVRBTuA6hd6v5xHlHSwc4qEtr-ZeB2TdPNK6EB-FMuNTAV2f2332NTY22-kDypMNaFG9NhzDwp_jDWC48w1Q0AUD8Kj5fNmqCAHpk64sxoV5U64keTcBrEmrFQy4OiMLket7qvqgfTy8SHL1arDFVQv7xg_brYP4b39IOIcqntqSxWG7kRIRhJaX0YCs_XzcvHERxqd7wLtnTl71f3Dh93YbPfqXCLQFZNlAJOzZd1rCopnjPKile59Ke3EMUE_ykgF-BhSWVGwqxarkUNg-3__jspU6pLvLY1Gy4Uuz33wPqG5kDbwFv4zVbLPR5F5I1OwLJBBQNoXA2Sr0Oi3_ihz1qpLDU8aWMQr4FyUElutvdvJQ1ft4WPAxsxkSkL-ySZwy5ppDmve3TbpRrPbAjq6tqY2l9J2t-r0nLnlazHQAfhP72HvfLTpQEAAqlOpS4aIoJhMK9PzRKJXrgCyDbRtW3fnv1XzPA5Kf4OOeg"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Cookie": f".ROBLOSECURITY={ROBLOSECURITY}"
}

def get_csrf():
    try:
        r = requests.post("https://auth.roblox.com/v2/logout", headers=headers)
        return r.headers.get("x-csrf-token")
    except:
        return None

def run_deletion():
    try:
        auth_res = requests.get("https://users.roblox.com/v1/users/authenticated", headers=headers)
        auth_res.raise_for_status()
        user_id = auth_res.json()['id']
        print(f"Authenticated as: {user_id}")
    except Exception as e:
        print(f"Auth failed: {e}. Check cookie.")
        return

    while True:
        print("\nFetching badges...")
        badges_res = requests.get(f"https://badges.roblox.com/v1/users/{user_id}/badges?limit=100", headers=headers)
        badges = badges_res.json().get('data', [])

        if not badges:
            print("Inventory cleared!")
            break

        for b in badges:
            badge_id = b['id']
            success = False

            while not success:  # Loop retry until success
                token = get_csrf()
                current_headers = headers.copy()
                current_headers["X-CSRF-TOKEN"] = token

                res = requests.delete(f"https://badges.roblox.com/v1/user/badges/{badge_id}", headers=current_headers)

                if res.status_code == 200:
                    print(f"[SUCCESS] Deleted: {badge_id}")
                    success = True
                elif res.status_code == 429:
                    print(f"[RATE LIMIT] Waiting 5s for {badge_id}...")
                    time.sleep(5)
                else:
                    print(f"[RETRYING] {badge_id} - Status {res.status_code}")
                    time.sleep(1)

            # Minimal gap to prevent instant 429 triggers
            time.sleep(0.5)

run_deletion()